In [1]:
from pathlib import Path
from datetime import datetime
import json
from io import BytesIO
from zipfile import ZipFile, BadZipFile
from tqdm import tqdm
import requests
import time

import pandas_datareader.data as web
import pandas as pd

from pprint import pprint

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import warnings

In [2]:
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

## SEC Edgar Data

- EDGAR acess policy: https://www.sec.gov/os/accessing-edgar-data
- EDGAR FSN data: https://www.sec.gov/about/divisions-offices/division-economic-risk-analysis/data/financial-statement-and-notes-data-set
    -  quarterly `2009q1`~ `2020-q3`, monthly: `2020-10`~
- EDGAR FS data: https://www.sec.gov/dera/data/financial-statement-data-sets
    - quarterly `2009q1`~ 

In [3]:
def download_FSN_from_sec(url, path):

        # Declare user agent in request headers
    headers = {
        'User-Agent': 'xikest12@gmail.com',
        'Accept-Encoding': 'gzip, deflate',
        'Host': 'www.sec.gov'
    }
    # Download and save file
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            with open('downloaded_file.zip', 'wb') as f:
                f.write(response.content)
                
            # Decompress and save
            with ZipFile(BytesIO(response.content)) as zip_file:
                for file in zip_file.namelist():
                    local_file = path / file
                    if local_file.exists():
                        continue
                    with local_file.open('wb') as output:
                        for line in zip_file.open(file).readlines():
                            output.write(line)

            # Sleep to comply with request rate limit
            time.sleep(0.1)  # Adjust as needed
        else:
            print(f"Failed to download file. Status code: {response.status_code}")
            print("Response content:", response.content)
    except BadZipFile:
        print(f'\nBad zip file: {url}\n')
        pass


def download_SEC_reports(data_path='data', start_date='2009', end_date='2023-11-30', freq='Q', type='fsn'):
    periods = []
    SEC_URL = 'https://www.sec.gov/'
    
    if type == 'fsn':
        FSN_PATH = 'files/dera/data/financial-statement-and-notes-data-sets/'
    elif type == 'fs':
        FSN_PATH = 'files/dera/data/financial-statement-data-sets/'
        
    if freq == 'Q':
        periods = [(d.year, d.quarter) for d in pd.date_range(start_date, end_date, freq='Q')]
    elif freq == 'M':
        periods = [(d.year, d.month) for d in pd.date_range(start_date, end_date, freq='M')]
    else:
        raise ValueError("Invalid frequency. Please use 'Q' for quarters or 'M' for months.")

    for yr, time_period in tqdm(periods):
        # Set (and create) directory
        if type == 'fsn':
            if freq == 'Q':
                path = data_path / f'{yr}_{time_period}q' / 'source'
                filing = f'{yr}q{time_period}_notes.zip'
            elif freq == 'M':
                path = data_path / f'{yr}_{time_period:02}' / 'source'
                filing = f'{yr}_{time_period:02}_notes.zip'
        elif type == 'fs':
            path = data_path / f'{yr}_{time_period}q' / 'source'
            filing = f'{yr}q{time_period}.zip'
            

        if not path.exists():
            path.mkdir(parents=True)

        url = SEC_URL + FSN_PATH + filing
        # print(url)
        download_FSN_from_sec(url, path)
    print(data_path.cwd())

In [4]:
DATA_STORAGE = "E:/mlft/data"
type ='fsn'
data_path = 'sec-filings/'+type
data_path = DATA_STORAGE/Path(data_path)
if not data_path.exists():
    data_path.mkdir()

In [9]:
# 분기별 데이터
download_SEC_reports(data_path, '2009', '2020-11-30', freq='Q', type=type)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [03:14<00:00, 16.23s/it]

E:\mlft\02_market_and_fundamental_data\04_sec_edgar


In [ ]:
# 월간 데이터
data_path = download_SEC_reports(data_path, '2020-10-1', '2023-11-30', freq='M')

  0%|                                                                                                                                                                                                                                                                                       | 0/38 [00:00<?, ?it/s]

https://www.sec.gov/files/dera/data/financial-statement-and-notes-data-sets/2020_10_notes.zip


## Save to parquet

일부 `txt.tsv`에서 결함이 있는 텍스트 라인이 있을 수 있고, 무시하고 진행 됨

In [11]:
for f in tqdm(sorted(list(data_path.glob('**/*.tsv')))):
    # set (and create) directory
    parquet_path = f.parent.parent / 'parquet'
    if not parquet_path.exists():
        parquet_path.mkdir(parents=True)    

#     # write content to .parquet
    file_name = f.stem  + '.parquet'
    if not (parquet_path / file_name).exists():
        try:
            df = pd.read_csv(f, sep='\t', encoding='latin1', low_memory=False, on_bad_lines='warn')
            df.to_parquet(parquet_path / file_name)
        except Exception as e:
            print(e, ' | ', f)
        # optional: uncomment to delete original .tsv
#         else:
            # f.unlink

100%|███████████████████████████| 680/680 [01:25<00:00,  7.95it/s]


---